# Week 1 — Research Question

**Author:** Hamza Khan**Repo:** `flyrank-ml-internship-starter`
**Date:** 2026-07-10

This notebook frames my capstone question before I build anything. It follows
`docs/ml-intern-dataset-and-lane-guide.md`. Everything below is written against the
30,000-row anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`); the
lane choice is provisional and I can revisit it up to the end of Week 4.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Hamzakhan0332/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    os.chdir("..")  # move from work/notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /home/claude/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

**Lane: 4 — CTR / Engagement Opportunity Scoring**

> *Which visible pages under-capture clicks or engagement and deserve metadata, content,
> or monitoring review?*

I looked at two candidates before settling here: Lane 2 (Refresh/Content Opportunity
Scoring) and Lane 4. Lane 2 already has a working baseline-vs-model comparison in this
repo's own pipeline (random forest beats the rule-based baseline, precision@50 0.740 vs
0.240), which is a great worked example — but that also means less of the "figure it out
myself" work is left for me to do.

Lane 4 is less worked-over and has a genuinely interesting hook in this starter data: a
large pool of pages hold decent search position *and* real impression volume, but their
click-through rate varies enormously even within the *same* position tier. That spread —
not just "low CTR" in isolation — is what makes this a modeling/analysis problem rather
than a one-line SQL filter. I show the numbers behind this in Section 3.

## 2. The question: decision, action, cost of a wrong call

**Unit of analysis:** one page (`content_id`) for one client, summarized over its trailing
90-day window.

**Question:** Among pages that already hold real search visibility (enough impressions,
a page-one-ish or striking-distance position), which ones are capturing meaningfully fewer
clicks or engagement than peers in the same position tier — and are therefore worth a
human review?

**Output:** a ranked review queue — a position-adjusted CTR/engagement gap score per page,
plus a reason code (e.g. *low CTR for position*, *high impressions but weak engagement*)
and the raw numbers a reviewer would want to see (impressions, clicks, CTR, avg position,
engagement rate, scroll rate).

**Action a human takes:** a content editor works down the queue and, for each flagged
page, checks the title/meta description, on-page intent match, and snippet structure —
and decides whether to rewrite, restructure, or simply monitor. This is decision-support:
the model ranks candidates for review, it does not publish changes on its own.

**Cost of a wrong call:**
- *False positive* (flagged, but nothing was actually wrong): an editor spends limited
  review time on a page that didn't need it, and — worse — a poorly-motivated rewrite
  could make a fine page slightly worse. Bounded cost, but not free.
- *False negative* (a real gap, never surfaced): the page keeps quietly under-capturing
  clicks it could otherwise get; the missed upside is the number of extra clicks or
  sessions the page would earn if it matched its position-tier peers.
- Because a human reviews every candidate before anything changes, the worst case is
  wasted attention, not a broken page — which is exactly why minimum-volume filters and
  a defensible baseline still matter: they decide how much of a limited reviewer's week
  gets spent chasing noise.

**Why data/ML helps at all:** with tens of thousands of pages and a handful of reviewer
hours per week, nobody can eyeball every page. A simple rule like "flag anything with
CTR < 0.5%" ignores that expected CTR differs by position — Section 3 shows position alone
barely explains click performance, so a fixed threshold either over-flags high-position
pages or misses real gaps at weaker positions. A model or residual score that adjusts for
position (and possibly intent/content type) turns "low CTR" into "lower than expected for
where this page already sits," which is a much more defensible, reviewable claim.

## 3. Quick look at the data (real numbers)

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same starter-pipeline filters as scripts/01_prepare_features.py
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates("content_id")

print(f"Rows after starter filters: {len(df):,}")
df.shape

Rows after starter filters: 30,000


(30000, 44)

**Number 1 — the review pool is real, not tiny.** Pages that already hold meaningful
search visibility *and* a competitive position are exactly the ones a CTR-gap analysis
should focus on (a page with 5 impressions or buried on page 4 isn't a CTR story).

In [3]:
visible = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
]
print(f"Pages with >=500 impressions/90d and position 1-20: {len(visible):,} "
      f"({len(visible)/len(df):.1%} of the filtered dataset)")

Pages with >=500 impressions/90d and position 1-20: 12,023 (40.1% of the filtered dataset)


**Number 2 — most of that pool looks like it's under-capturing clicks by a simple
threshold.** Using the lane guide's own `low_ctr_visible_page` rule (CTR < 0.5 within
that pool):

In [4]:
low_ctr = visible[visible["ctr"] < 0.5]
print(f"Low-CTR candidates: {len(low_ctr):,} ({len(low_ctr)/len(visible):.1%} of the visible pool)")

Low-CTR candidates: 9,759 (81.2% of the visible pool)


That 81% figure is suspiciously high for a single fixed threshold — which is itself a
useful, if uncomfortable, finding: a flat CTR cutoff flags almost everyone, so it can't be
the final rule. That's exactly why the next number matters.

**Number 3 — position alone barely explains click performance, so a position-adjusted
gap score has real work to do.** Two views of the same point: the overall correlation
between position and clicks is weak, and CTR spreads widely even *within* one position
tier.

In [5]:
corr = df[["avg_position", "clicks_90d"]].corr().iloc[0, 1]
print(f"Correlation(avg_position, clicks_90d): {corr:.3f}")

print("\nCTR spread within the 'page_1' position tier (visible pool):")
page1 = visible[visible["position_tier"] == "page_1"]["ctr"]
print(page1.describe()[["min", "25%", "50%", "75%", "max"]].round(2))

Correlation(avg_position, clicks_90d): -0.099

CTR spread within the 'page_1' position tier (visible pool):
min    0.00
25%    0.12
50%    0.24
75%    0.44
max    5.42
Name: ctr, dtype: float64


Two pages can hold essentially the same position and land anywhere from near-zero CTR
to well above the tier median — position explains very little of that spread on its own.
That's the gap a position-adjusted score is meant to capture, rather than treating a flat
CTR cutoff as the answer.

## 4. Careful words: what I can and can't claim

**I can say (observed / measured):**
- This starter slice shows a large pool (12,023 pages) with real visibility and position,
  and wide CTR variation within the same position tier.
- A flat CTR threshold flags the large majority of the visible pool, which suggests the
  threshold itself — not just the pages — needs work.
- Position alone is a weak predictor of clicks in this data (r ≈ -0.10); other factors
  (intent match, title/snippet, content type) plausibly matter more.

**I cannot yet say:**
- That any specific page's low CTR is caused by its title, meta description, or content —
  that requires review, not just a number.
- That fixing a flagged page will increase its CTR — that's a causal claim I have not
  tested (see `DATA_USE.md` and lane guide §14 on public-safe claims).
- That this pattern holds on the full warehouse release — this is a 30,000-row anonymized
  sample from one client mix; the real release (`fact_content_daily_performance`,
  78.8M rows) may look different, and I still need client-level minimum-volume filters
  before I trust any single client's numbers.
- Anything about *why* CTR varies without cross-checking low-volume noise, consolidation
  (a sibling URL absorbing clicks), or seasonality — lane guide §7's cautions apply here
  just as much as they do to decline lanes.

I'll keep all outputs in `work/` framed as observed/measured/directional/decision-support,
per `README.md` §"Rules of the road" and `DATA_USE.md` — no client-identifying details, no
causal language, until there's a design that supports it.

## 5. Self-check

- [x] Picked one predefined lane (Lane 4: CTR / Engagement Opportunity Scoring) and said why.
- [x] Named the unit of analysis (one page, 90-day window).
- [x] Named the decision (which pages to review first) and the action a human takes
      (title/meta/intent/engagement review, not an automated edit).
- [x] Named the cost of a wrong call in both directions (wasted review time vs. missed
      upside), and why the human-in-the-loop design bounds the downside.
- [x] Showed 3 real numbers from the starter dataset: pool size (12,023), flat-threshold
      flag rate (81.2%), and the weak position→clicks correlation (-0.10) plus within-tier
      CTR spread.
- [x] Explained why this isn't just "train a model": the interesting problem is defining
      an *expected* CTR per position (and later, per intent/content type) so "low CTR" is
      meaningful relative to peers, not an arbitrary flat cutoff — that's a gap/residual
      analysis question before it's a modeling question.
- [x] Used careful language — no causal claims, no algorithm claims, no client-identifying
      detail; flagged what full-warehouse validation and minimum-volume filtering I still owe.
- [ ] Lane is provisional — open to revisiting through Week 4 if the position-adjusted
      gap score turns out thin once I dig into content_type / intent splits.